# C4 - Custom ReAct Agent with Configurable Personas
**Joey Saade | 202305993**

Use case: the Harborlight study advisor helps students and families match needs to published tutoring services. Tools: `lookup_business_fact` and `suggest_service`. Constraints: no invented prices, availability, bookings, or outcomes.

In [ ]:
%pip install -r requirements.txt

## Manual LangGraph ReAct loop
`agent.py` wires `think -> act -> think -> finish`. The model emits JSON with either an action and argument or an answer. The action node executes a local tool and records the observation. A four-call cap limits loops. No prebuilt agent executor is used.

In [ ]:
from agent import PERSONAS, INSTRUCTIONS, TOOL_DESCRIPTIONS, lookup_business_fact, suggest_service, ask
print(PERSONAS)
print(INSTRUCTIONS)
print(TOOL_DESCRIPTIONS)

## Persona and configuration design
The friendly advisor uses warm, plain language with low thinking and a 1024 token cap. The careful expert uses measured vocabulary with medium thinking and a 1536 token cap. The model configurations match the intended conversational styles.

In [ ]:
print(lookup_business_fact("programming"))
print(suggest_service("I need help with calculus"))

## Prompt experiments
Run `python evaluate.py` for 24 live trials: two personas, four prompt variants, three cases. The variants are few-shot system, zero-shot system, a brief private decision check, and in-message instructions. Inspect `evaluation_live.jsonl` for answers and tool traces. The automatic keyword check is only a screening aid; judge usefulness and factual accuracy manually.

In [ ]:
import os
if os.getenv("GEMINI_API_KEY"):
    result = ask("Can you help with introductory Python?", "friendly_advisor")
    print(result["answer"])
    print(result["trace"])
else:
    print("Add GEMINI_API_KEY to .env, restart the kernel, and rerun.")

## Reflection

**Most helpful or natural persona:** The friendly advisor is expected to be more natural for a discovery conversation; the careful expert may be more reliable for factual questions. This is a hypothesis until live responses are scored.

**Best prompt and configuration:** The few-shot system variant with careful expert settings is the initial candidate because it shows the action format and uses a higher reasoning budget. Compare it with all 24 live trials before claiming a winner.

**Reasoning and tools:** The graph makes every tool request and observation visible in the trace. The model must use a tool for business claims; inspect the trace to verify that it does.

**Effect of instructions:** The separate instructions specify JSON output, a worked example, tool use, and factual limits. Compare zero-shot and few-shot runs to see whether these reduce malformed actions and invented facts.

**Biggest challenge:** Converting model text into dependable actions. JSON mode, explicit tool names, a bounded cycle, and a safe fallback reduce that risk. The exact performance still depends on live model responses.